In [1]:
# Identifiers (RQx, Hx, Px, Gx, Fx, Tx) are defined in vpir_bench_spec.md: section 2 for the research questions,
# section 11 for the tests behind each gate, section 12 for phases, gates and figures. Rx reproduction targets are
# in docs/reproduction_targets.md, and numbered decisions in plan.md.
import json, sys
from collections import Counter, defaultdict
from dataclasses import asdict, replace
from pathlib import Path

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.harness import recovery as R

# RQ4 in one day (decided 2026-11-11). The spec's counts take an expected 1115.5 h on this Mac
# (notebook 09), so every arm runs at one cell, 128 MiB x 1 KiB: the only focus size where S1, S2 and S3
# fit (notebook 08), with the record size the spec keeps when it shrinks RQ4. S3 retrieves one bit.
# S4 has no focus cell, so it runs at 64 KiB, its largest affordable size (notebook 06).
MiB, KiB = 1 << 20, 1 << 10
TRIALS = 100            # trials per (arm, fault, policy); the spec's count
TRIALS_SLOW = 30        # S3 and S4, whose queries take 1 to 2 s
OFFLINE_REPS = {"S1": 3, "S2": 3, "S0-go": 10, "S0-cpp": 10}   # each repetition is a full offline phase
QUERIES = 1000          # sweep queries per rate; the spec's count
QUERIES_SLOW = 50
RE_PREPROCESS_RATE, RE_PREPROCESS_QUERIES = 0.1, 50
MIN_CHEATS = 5
SLOW = {"S3", "S4"}

def day_plan(runs):
    day = []
    for r in runs:
        if r.kind == "trials":
            # At fault rate 1 the same server cheats on every attempt, so a re_preprocess trial is always
            # three offline phases ending aborted, which P7 already checks; its recoveries come from the sweep.
            if r.policy != "re_preprocess":
                day.append(replace(r, reps=TRIALS_SLOW if r.arm in SLOW else TRIALS))
        elif r.kind == "offline":
            day.append(replace(r, reps=OFFLINE_REPS[r.arm]))
        else:
            q = QUERIES_SLOW if r.arm in SLOW else QUERIES
            if r.fault_prob == 0:
                if r.policy == "abort":          # nothing is rejected at f = 0: one honest control per arm
                    day.append(replace(r, reps=q))
            elif r.policy == "re_preprocess":
                if r.fault_prob == RE_PREPROCESS_RATE:
                    day.append(replace(r, reps=RE_PREPROCESS_QUERIES))
            elif q * r.fault_prob >= MIN_CHEATS:  # fewer expected cheats than this says nothing f = 0 does not
                day.append(replace(r, reps=q))
    return day

cfg = R.load_config(R.DEFAULT_CONFIG)
day = day_plan(R.plan(cfg, db_sizes=[128 * MiB], record_sizes=[1024]) + R.plan(cfg, only=["S4"], db_sizes=[64 * KiB]))

table = R.load_estimates(repo / "results/raw/grid-feasibility-20261111T064956Z/estimates.jsonl")
table[("S4", 64 * KiB)] = {0: {"offline_s": 22.5, "query_s": 1.21}}   # notebook 06, cell 2
expected = sum(R.expected_run_s(r, table) for r in day)
print(f"{len(day)} Runs, {sum(r.expected_records for r in day)} adapter repetitions, expected {expected / 3600:.2f} h "
      "(S1, S2 and S0-cpp at 1 KiB priced from their 8 KiB times, so an upper estimate)")
per = Counter((r.arm, r.kind) for r in day)
for arm in R.BINARY:
    print(f"  {arm:7s} " + "  ".join(f"{k} {per[(arm, k)]:2d}" for k in R.KINDS)
          + f"  rates {sorted({r.fault_prob for r in day if r.arm == arm and r.kind == 'sweep'})}")

116 Runs, 47028 adapter repetitions, expected 5.93 h (S1, S2 and S0-cpp at 1 KiB priced from their 8 KiB times, so an upper estimate)
  S0-go   trials  4  offline  1  sweep  5  rates [0.0, 0.01, 0.1, 0.5, 1.0]
  S0-cpp  trials  4  offline  1  sweep  5  rates [0.0, 0.01, 0.1, 0.5, 1.0]
  S1      trials  8  offline  2  sweep 10  rates [0.0, 0.01, 0.1, 0.5, 1.0]
  S2      trials  8  offline  2  sweep 10  rates [0.0, 0.01, 0.1, 0.5, 1.0]
  S3      trials  8  offline  0  sweep  7  rates [0.0, 0.1, 0.5, 1.0]
  S4      trials  8  offline  0  sweep  7  rates [0.0, 0.1, 0.5, 1.0]
  M0      trials  4  offline  0  sweep  5  rates [0.0, 0.01, 0.1, 0.5, 1.0]
  M1      trials  8  offline  0  sweep  9  rates [0.0, 0.01, 0.1, 0.5, 1.0]


In [2]:
# Hours of adapter runs, serially, each under caffeinate, the 14 GiB cap and a hygiene watch. The run stops
# if the Mac sleeps, loses AC power or has its lid closed. Rerunning this cell resumes the same run id.
RUN_ID = "rq4-day-20261111"
ex = R.prepare(day, cfg, R.DEFAULT_CONFIG, run_id=RUN_ID)
plan_file = ex.out_dir / "plan.jsonl"
if not plan_file.exists():
    plan_file.write_text("".join(json.dumps(asdict(r)) + "\n" for r in day))
assert [json.loads(l) for l in plan_file.read_text().splitlines()] == [asdict(r) for r in day], "the plan changed"
status = R.run_and_report(day, ex)

run rq4-day-20261111: 116 Runs into /Users/imanchakraborty/cs4406-sem-project/results/raw/rq4-day-20261111


[1/116] ok trials-S0-go-db134217728-rb1024-flip_answer_bit-abort-p1 (13 s) 


[2/116] ok trials-S0-go-db134217728-rb1024-wrong_record-abort-p1 (15 s) 


[3/116] ok trials-S0-go-db134217728-rb1024-stale_record-abort-p1 (15 s) 


[4/116] ok trials-S0-go-db134217728-rb1024-drop_server-abort-p1 (13 s) 


[5/116] ok trials-S0-cpp-db134217728-rb1024-flip_answer_bit-abort-p1 (39 s) 


... (108 outputs left out)

[114/116] ok sweep-S4-db65536-rb0-wrong_record-switch_replica-p0_1-cheat (101 s) 


[115/116] ok sweep-S4-db65536-rb0-wrong_record-switch_replica-p0_5-cheat (193 s) 


[116/116] ok sweep-S4-db65536-rb0-wrong_record-switch_replica-p1-cheat (531 s) 


116 of 116 Runs settled (complete, oom or timeout) in 24461 s


In [3]:
rows = [json.loads(l) for l in (ex.out_dir / "manifest.jsonl").read_text().splitlines()]
last = {r["name"]: r for r in rows}                     # a resumed Run's latest attempt
outcomes = Counter(r["outcome"] for r in last.values())
print(f"{len(last)} of {len(day)} Runs attempted: {dict(outcomes)}; citable {sum(r['citable'] for r in last.values())}")
by_arm = defaultdict(Counter)
for r in last.values():
    by_arm[r["run"]["arm"]][r["outcome"]] += 1
for arm, c in by_arm.items():
    print(f"  {arm:7s} {dict(c)}")
for r in last.values():
    if r["outcome"] != "ok":
        print(f"  not ok: {r['name']} {r['outcome']} records {r['records']}/{r['run']['reps'] + r['run']['warmup']}")
print(f"adapter wall time {sum(r['wall_s'] for r in last.values()) / 3600:.2f} h")

116 of 116 Runs attempted: {'ok': 116}; citable 116
  S0-go   {'ok': 10}
  S0-cpp  {'ok': 10}
  M0      {'ok': 9}
  S1      {'ok': 20}
  S2      {'ok': 20}
  S3      {'ok': 15}
  M1      {'ok': 17}
  S4      {'ok': 15}
adapter wall time 6.79 h
